# MangaForge AI — Kaggle pipeline
Enable Internet in Kaggle settings. Search metadata is not permission to download or train. Verify original source and AI-training rights independently. Attach reviewed image datasets and manifests under `/kaggle/input`. Outputs in `/kaggle/working` must be saved as a notebook version.

In [ ]:
from pathlib import Path
import subprocess,sys,csv,json
ROOT=Path('/kaggle/working/mangaforge');ROOT.mkdir(parents=True,exist_ok=True)
REPO=Path('/kaggle/working/MangaForge-AI-')
BRANCH='feature/licensed-dataset-importer'
if (REPO/'.git').exists():
    subprocess.run(['git','-C',str(REPO),'fetch','--depth','1','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','-B',BRANCH,'FETCH_HEAD'],check=True)
else:
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/xynewageyx/MangaForge-AI-.git',str(REPO)],check=True)
print('Ready',REPO)

## 1. Openverse discovery
Find CC0/PDM candidates. No automatic approval; inspect original pages. Requests can be rate-limited.

In [ ]:
CANDIDATES=ROOT/'openverse_candidates.csv'
subprocess.run([sys.executable,str(REPO/'tools/discover_openverse.py'),'--output',str(CANDIDATES),'--pages','1'],check=True)
print('Review candidates:',CANDIDATES)

## 2. Local image inventory
Attach an image dataset. All permissions default to no. Review licenses, commercial use and AI-training rights separately.

In [ ]:
SOURCE_DIR=Path('/kaggle/input/mangaforge-images')
MANIFEST=ROOT/'manifest.csv'
if SOURCE_DIR.is_dir():
    subprocess.run([sys.executable,str(REPO/'tools/build_candidate_manifest.py'),'--input',str(SOURCE_DIR),'--output',str(MANIFEST)],check=True)
    print('Review manifest:',MANIFEST)
else: print('Attach an image dataset and edit SOURCE_DIR')

## 3. Rights-gated import
Attach the reviewed manifest. Source image paths must exist in this Kaggle session. The importer checks permission fields but cannot verify legal claims.

In [ ]:
REVIEWED_MANIFEST=Path('/kaggle/input/mangaforge-reviewed-manifest/manifest.csv')
APPROVED=ROOT/'approved_dataset'
if REVIEWED_MANIFEST.is_file():
    subprocess.run([sys.executable,str(REPO/'tools/licensed_dataset_importer.py'),'--manifest',str(REVIEWED_MANIFEST),'--output',str(APPROVED)],check=True)
else: print('No reviewed manifest; import skipped')

## 4. Quality control and provisional classification
Technical quality and near-duplicate analysis; category suggestions based on filenames, not image understanding.

In [ ]:
try: import PIL
except ImportError: subprocess.run([sys.executable,'-m','pip','install','Pillow'],check=True)
if (APPROVED/'accepted.csv').is_file():
    subprocess.run([sys.executable,str(REPO/'tools/audit_dataset.py'),'--input',str(APPROVED/'accepted.csv'),'--output',str(APPROVED/'quality_report.csv')],check=True)
    from importlib.machinery import SourceFileLoader
    cls=SourceFileLoader('classify',str(REPO/'tools/build_candidate_manifest.py')).load_module()
    with (APPROVED/'accepted.csv').open(encoding='utf-8',newline='') as f: records=list(csv.DictReader(f))
    with (ROOT/'training_review.csv').open('w',encoding='utf-8',newline='') as f:
        fields=['imported_path','category_suggestion','category_verified','caption_verified','training_approved']
        w=csv.DictWriter(f,fieldnames=fields);w.writeheader()
        for r in records:w.writerow({'imported_path':r['imported_path'],'category_suggestion':cls.category_for(Path(r['path']).name),'category_verified':'','caption_verified':'','training_approved':'no'})
    print('Review quality_report.csv and training_review.csv')
else: print('No approved images; audit skipped')

## 5. Training preparation
Attach a human-reviewed training_review.csv. Only reviewed, approved images with verified captions are exported. No model training is run.

In [ ]:
LABELS=Path('/kaggle/input/mangaforge-reviewed-labels/training_review.csv')
if LABELS.is_file() and (APPROVED/'accepted.csv').is_file():
    with (APPROVED/'accepted.csv').open(encoding='utf-8',newline='') as f: allowed={str(Path(r['imported_path']).resolve()) for r in csv.DictReader(f)}
    count=0
    with LABELS.open(encoding='utf-8-sig',newline='') as f, (ROOT/'train.jsonl').open('w',encoding='utf-8') as out:
        for r in csv.DictReader(f):
            p=Path(r.get('imported_path','')).resolve()
            cat=r.get('category_verified','').strip();caption=r.get('caption_verified','').strip()
            if str(p) in allowed and p.is_file() and r.get('training_approved','').strip().lower()=='yes' and cat and caption:
                out.write(json.dumps({'image':str(p),'category':cat,'caption':caption},ensure_ascii=False)+'\n');count+=1
    print('Training records:',count)
else: print('No reviewed labels; training export skipped')